In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
input_root = "/content/drive/MyDrive/Images"


In [ ]:
import os
import cv2

CONVERSIE POZE ---separare in 2 foldere

In [ ]:
import os
import cv2

input_root = "/content/drive/MyDrive/Images"

out_images = "/content/drive/MyDrive/dataset/images"
out_masks  = "/content/drive/MyDrive/dataset/masks"

os.makedirs(out_images, exist_ok=True)
os.makedirs(out_masks, exist_ok=True)

count = 0

for root, _, files in os.walk(input_root):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            path = os.path.join(root, file)
            img = cv2.imread(path)

            if img is None:
                continue

            h, w, _ = img.shape
            if h != 512 or w < 1024:
                continue

            # nume unic
            rel_path = os.path.relpath(root, input_root)
            safe_name = rel_path.replace(os.sep, "_") + "_" + file

            oct_img  = img[:, :512]
            mask_img = img[:, 512:1024]

            cv2.imwrite(os.path.join(out_images, safe_name), oct_img)
            cv2.imwrite(os.path.join(out_masks,  safe_name), mask_img)

            count += 1

print(f"Separare completă  {count} imagini procesate")


Separare completă  3859 imagini procesate


In [ ]:
import os
import cv2
import numpy as np

In [ ]:
print("Images:", len(os.listdir("/content/drive/MyDrive/dataset/images")))
print("Masks :", len(os.listdir("/content/drive/MyDrive/dataset/masks")))


Images: 3859
Masks : 3859


In [ ]:
mask_dir = "/content/drive/MyDrive/dataset/masks"
out_dir  = "/content/drive/MyDrive/dataset/masks_labels"

In [ ]:
import os
import cv2
import numpy as np

mask_dir = "/content/drive/MyDrive/dataset/masks"
out_dir  = "/content/drive/MyDrive/dataset/masks_labels"

os.makedirs(out_dir, exist_ok=True)

# BGR !!!
color_map = {
    (0,   0, 255): 1,  # rosu
    (0, 255,   0): 2,  # verde
    (255, 0,   0): 3,  # albastru
    (0, 255, 255): 4   # galben
}

converted = 0

for file in os.listdir(mask_dir):
    path = os.path.join(mask_dir, file)
    mask = cv2.imread(path)

    if mask is None:
        continue

    label = np.zeros(mask.shape[:2], dtype=np.uint8)

    # culorile pure
    for bgr, cls in color_map.items():
        matches = np.all(mask == bgr, axis=-1)
        label[matches] = cls

    # fundal (aproape negru) → clasa 0
    gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    label[gray < 30] = 0   # toleranță fundal

    out_path = os.path.join(out_dir, os.path.splitext(file)[0] + ".png")
    cv2.imwrite(out_path, label)
    converted += 1

print("Masti convertite:", converted)


Masti convertite: 3859


In [ ]:
# verificam clase
sample = os.listdir(out_dir)[0]
lbl = cv2.imread(os.path.join(out_dir, sample), 0)
print("Clase găsite:", np.unique(lbl))


Clase găsite: [0 2 4]


In [ ]:
import os
from sklearn.model_selection import train_test_split

img_dir = "/content/drive/MyDrive/dataset/images"
mask_dir = "/content/drive/MyDrive/dataset/masks_labels"

files = sorted(os.listdir(img_dir))

train_files, val_files = train_test_split(
    files, test_size=0.2, random_state=42
)

print("Train:", len(train_files))
print("Val  :", len(val_files))


Train: 3087
Val  : 772


In [ ]:
import torch
from torch.utils.data import Dataset
import cv2
import os
import numpy as np

class OCTDataset(Dataset):
    def __init__(self, files, img_dir, mask_dir):
        self.files = files
        self.img_dir = img_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        img = cv2.imread(os.path.join(self.img_dir, fname), 0)
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).unsqueeze(0)

        mask = cv2.imread(os.path.join(self.mask_dir, fname), 0)
        mask = torch.from_numpy(mask).long()

        return img, mask


In [ ]:
import torch.nn as nn
import torch

class UNet(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()

        def CBR(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1),
                nn.BatchNorm2d(o),
                nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1),
                nn.BatchNorm2d(o),
                nn.ReLU(inplace=True),
            )

        self.enc1 = CBR(1, 64)
        self.enc2 = CBR(64, 128)
        self.enc3 = CBR(128, 256)
        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = CBR(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = CBR(128, 64)

        self.final = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))

        x = self.up2(x3)
        x = self.dec2(torch.cat([x, x2], dim=1))

        x = self.up1(x)
        x = self.dec1(torch.cat([x, x1], dim=1))

        return self.final(x)


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_onehot = torch.nn.functional.one_hot(
            targets, num_classes=probs.shape[1]
        ).permute(0,3,1,2).float()

        dims = (0,2,3)
        intersection = torch.sum(probs * targets_onehot, dims)
        union = torch.sum(probs + targets_onehot, dims)

        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


In [ ]:
ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

def loss_fn(pred, mask):
    return ce_loss(pred, mask) + dice_loss(pred, mask)


In [ ]:
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = OCTDataset(train_files, img_dir, mask_dir)
val_ds   = OCTDataset(val_files, img_dir, mask_dir)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4)

model = UNet(n_classes=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)



In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    num_workers=4,
    pin_memory=True
)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
!pip install albumentations


In [ ]:
from tqdm import tqdm

epochs = 30
batch_size = 4
lr = 1e-4

torch.backends.cudnn.benchmark = True


for epoch in range(epochs-6):
    model.train()
    train_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1:02d}/{epochs} [Train]",
        leave=False
    )

    for img, mask in train_bar:
        img  = img.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        pred = model(img)
        loss = loss_fn(pred, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0

    val_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch+1:02d}/{epochs} [Val]",
        leave=False
    )

    with torch.no_grad():
        for img, mask in val_bar:
            img  = img.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)

            loss = loss_fn(model(img), mask)
            val_loss += loss.item()
            val_bar.set_postfix(loss=loss.item())

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}",
        flush=True
    )


torch.save(model.state_dict(), "/content/drive/MyDrive/OIMHS_segmentare_unet_oct.pth")


Epoch 01/30 | Train Loss: 0.8506 | Val Loss: 0.6423


Epoch 02/30 | Train Loss: 0.4939 | Val Loss: 0.4501


Epoch 03/30 | Train Loss: 0.4310 | Val Loss: 0.4269


Epoch 04/30 | Train Loss: 0.4093 | Val Loss: 0.4267


Epoch 05/30 | Train Loss: 0.3974 | Val Loss: 0.3969


...

Epoch 19/30 | Train Loss: 0.3602 | Val Loss: 0.3706


Epoch 20/30 | Train Loss: 0.3547 | Val Loss: 0.3658


Epoch 21/30 | Train Loss: 0.3542 | Val Loss: 0.3649


Epoch 22/30 | Train Loss: 0.3546 | Val Loss: 0.3721


Epoch 23/30 | Train Loss: 0.3539 | Val Loss: 0.3797


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/OIMHS_segmentare_unet_oct.pth")
